In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")
vs_indx = VectorSearchIndex(
    keyword_fields=['course'],
    mode= 'ivf',
    db_path='faq_vector.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
query =  "Can I still join the course after the start date?"
vector_query = model.encode(query)
results = vs_indx.search(
    vector_query,
    num_results=5,
    filter_dict = {"course": 'llm-zoomcamp'})

In [6]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the for

In [7]:
from Module01_AgenticRAG.rag_helper import RAGBase

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    # 1. Update the signature to accept 'course' as the second argument
    def search(self, query, course=None, num_results=5):
        query_vector = self.embedder.encode(query)

        target_course = course if course is not None else self.course
        filter_dict = {"course": target_course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("API_URL"),
)

In [9]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
CONTEXT:
""".strip()

vector_assistant = RAGVector(
    llm_client = openai_client,
    index = vs_indx,
    embedder = SentenceTransformer("all-MiniLM-L6-v2"),
    instructions = instructions
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
vector_assistant.rag_pipeline(query)

'Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [11]:
vs_indx.close()